# Shekina — Prototipo de orquestación (demo)
Este cuaderno demuestra la orquestación mínima desde una tabla canónica (JSON) con:
- s1: carga base (mirror.select_df simulado)
- s2: transmute (alchemist.transmute con Polars)
- s3: step con condición falsa → se salta
Al final se imprime un `run_log`.

In [7]:
# Setup: imports, registry, helpers
import json, os, time
from pathlib import Path
import pandas as pd
try:
    import polars as pl
except Exception:
    pl = None

# Registry minimal
class Registry:
    def __init__(self):
        self._f = {}
    def register(self, opcode, fn):
        self._f[opcode] = fn
    def resolve(self, opcode):
        return self._f[opcode]

registry = Registry()

# mirror.select_df (simulado, construye DataFrame local para demo)
def mirror_select_df(context, payload):
    sql = payload.get("sql", "")
    # Demo: parseo ultra simple para el SQL de ejemplo
    # Producción: usar Mirror real con engine y select_df
    if "UNION ALL" in sql:
        df = pd.DataFrame({"id": [1,2], "raw": ["A", "B"]})
    else:
        df = pd.DataFrame({"id": [1], "raw": ["A"]})
    context["base_df"] = df
    return {"rows": len(df)}

# alchemist.transmute (Polars si está disponible; fallback a pandas)
def alchemist_transmute(context, payload):
    inputs = payload.get("inputs") or ["base_df"]
    src = inputs[0]
    pdf = context[src]
    steps = payload.get("steps", [])

    if pl is not None:
        df_pl = pl.from_pandas(pdf)
        for st in steps:
            if st.get("type") == "CLEAN_IDENTIFIER":
                col = st["src"]
                tgt = st["target"]
                pattern = st.get("pattern", "[^A-Za-z0-9]")
                repl = st.get("replacement", "")
                df_pl = df_pl.with_columns(
                    pl.col(col).str.replace_all(pattern, repl).str.to_lowercase().alias(tgt)
                )
        out = df_pl.to_pandas()
    else:
        df_pd = pdf.copy()
        for st in steps:
            if st.get("type") == "CLEAN_IDENTIFIER":
                col = st["src"]
                tgt = st["target"]
                pattern = st.get("pattern", "[^A-Za-z0-9]")
                repl = st.get("replacement", "")
                df_pd[tgt] = df_pd[col].astype(str).str.replace(pattern, repl, regex=True).str.lower()
        out = df_pd

    context["clean_df"] = out
    return {"rows": len(out), "cols": list(out.columns)}

registry.register("mirror.select_df", mirror_select_df)
registry.register("alchemist.transmute", alchemist_transmute)

# helpers
def eval_condition(cond):
    # Demo: solo soporta {"==": [a,b]}
    if not cond:
        return True
    if "==" in cond:
        a, b = cond["=="]
        return a == b
    return True


In [8]:
# Cargar tabla canónica y compilar un DAG simple (toposort)
proc_path = Path("proceso_demo.json")
with open(proc_path, "r", encoding="utf-8") as f:
    process = json.load(f)

# índice por step id
by_id = {s["id"]: s for s in process}
name_map = {s["step"]: s["id"] for s in process}

# toposort sencillo (Kahn)
from collections import defaultdict, deque
deps = {s["id"]: [name_map.get(d, d) for d in s.get("depends_on", [])] for s in process}
rev = defaultdict(list)
indeg = {k:0 for k in deps}
for k, ds in deps.items():
    for d in ds:
        rev[d].append(k)
        indeg[k]+=1
q = deque([k for k,v in indeg.items() if v==0])
order = []
while q:
    u = q.popleft()
    order.append(u)
    for v in rev[u]:
        indeg[v]-=1
        if indeg[v]==0: q.append(v)

# agrupar por lotes (capas)
levels = []
level_set = set([k for k,v in deps.items() if len(v)==0])
used = set()
while level_set:
    levels.append(list(level_set))
    used |= level_set
    next_set = set()
    for u in level_set:
        for v in rev[u]:
            if v in used:
                continue
            if all(d in used for d in deps[v]):
                next_set.add(v)
    level_set = next_set

levels

[['s1'], ['s2'], ['s3']]

In [9]:
# Ejecutar por lotes y construir run_log
context = {}
run_log = []

def run_step(s):
    if not s.get("enabled", True):
        return {"status": "disabled"}
    if not eval_condition(s.get("condition")):
        return {"status": "skipped"}
    opcode = s["opcode"]
    fn = registry.resolve(opcode)
    payload = dict(s.get("payload", {}))
    # inyectar inputs si corresponde
    if "inputs" in s:
        payload["inputs"] = s["inputs"]
    t0 = time.time()
    res = fn(context, payload)
    dt = time.time() - t0
    return {"status": "ok", "result": res, "dt_s": round(dt,4)}

for lvl in levels:
    for sid in lvl:
        s = by_id[sid]
        out = run_step(s)
        run_log.append({"step": s["step"], **out})

len(run_log), run_log[:2]

(3,
 [{'step': 'load_base', 'status': 'ok', 'result': {'rows': 2}, 'dt_s': 0.004},
  {'step': 'cleanup',
   'status': 'ok',
   'result': {'rows': 2, 'cols': ['id', 'raw', 'clean']},
   'dt_s': 0.008}])

In [10]:
# Inspección de artefactos y log final
print("context keys:", list(context.keys()))
if "clean_df" in context:
    display(context["clean_df"].head())
import pandas as pd
print(pd.DataFrame(run_log))

context keys: ['base_df', 'clean_df']


,id,raw,clean
0,1,A,a
1,2,B,b


        step   status                                       result   dt_s
0  load_base       ok                                  {'rows': 2}  0.004
1    cleanup       ok  {'rows': 2, 'cols': ['id', 'raw', 'clean']}  0.008
2  skip_demo  skipped                                          NaN    NaN


## Versión completa: Shekina → Aleya → Registry

En esta sección implementamos una mini versión de Aleya (runner) que compila el DAG desde la tabla canónica y ejecuta los pasos consultando el Registry. Shekina delega en Aleya la ejecución, y Aleya resuelve los opcodes con el Registry. Así vemos el flujo completo en el cuaderno.

In [11]:
# Mini-implementación de Aleya y uso desde Shekina
from dataclasses import dataclass
from typing import Any, Callable, Dict, List

@dataclass
class ShekinaOrchestrator:
    registry: Any
    aleya: Any

    def run_process_table(self, table: List[Dict[str, Any]]):
        # Validación mínima y delegación en Aleya
        return self.aleya.run(table, self.registry)

class AleyaRunner:
    def compile_dag(self, table: List[Dict[str, Any]]):
        # Reutilizamos la lógica de toposort usada arriba (sin duplicar mucho código)
        name_map = {s["step"]: s["id"] for s in table}
        from collections import defaultdict, deque
        deps = {s["id"]: [name_map.get(d, d) for d in s.get("depends_on", [])] for s in table}
        rev = defaultdict(list)
        indeg = {k:0 for k in deps}
        for k, ds in deps.items():
            for d in ds:
                rev[d].append(k)
                indeg[k]+=1
        q = deque([k for k,v in indeg.items() if v==0])
        order = []
        while q:
            u = q.popleft()
            order.append(u)
            for v in rev[u]:
                indeg[v]-=1
                if indeg[v]==0: q.append(v)
        # agrupar por niveles
        levels: List[List[str]] = []
        level_set = set([k for k,v in deps.items() if len(v)==0])
        used = set()
        while level_set:
            levels.append(list(level_set))
            used |= level_set
            next_set = set()
            for u in level_set:
                for v in rev[u]:
                    if v in used:
                        continue
                    if all(d in used for d in deps[v]):
                        next_set.add(v)
            level_set = next_set
        return levels, {s["id"]: s for s in table}

    def run(self, table: List[Dict[str, Any]], registry):
        levels, by_id_local = self.compile_dag(table)
        context: Dict[str, Any] = {}
        run_log: List[Dict[str, Any]] = []
        def eval_condition(cond):
            if not cond:
                return True
            if isinstance(cond, dict) and "==" in cond:
                a, b = cond["=="]
                return a == b
            return True
        for lvl in levels:
            for sid in lvl:
                s = by_id_local[sid]
                if not s.get("enabled", True):
                    run_log.append({"id": s["id"], "step": s["step"], "status": "disabled"})
                    continue
                if not eval_condition(s.get("condition")):
                    run_log.append({"id": s["id"], "step": s["step"], "status": "skipped"})
                    continue
                # Resolver callable
                fn = registry.resolve(s["opcode"]) if s.get("opcode") else None
                if fn is None:
                    run_log.append({"id": s["id"], "step": s["step"], "status": "unknown-opcode"})
                    continue
                payload = dict(s.get("payload", {}))
                # inyectar inputs si existen
                if "inputs" in s:
                    payload["inputs"] = s["inputs"]
                # Ejecutar y mapear outputs a context
                res = fn(context, payload)
                # si la función ya escribió outputs en el context (como en el demo), sólo registrar
                if s.get("outputs"):
                    # soporte para funciones que devuelven datos directos también
                    if isinstance(res, dict) and len(s["outputs"])==1 and s["outputs"][0] not in context:
                        context[s["outputs"][0]] = res
                run_log.append({"id": s["id"], "step": s["step"], "status": "ok"})
        return context, run_log, levels

# Ejecutar flujo completo
aleya = AleyaRunner()
shekina = ShekinaOrchestrator(registry=registry, aleya=aleya)

with open(proc_path, "r", encoding="utf-8") as f:
    table = json.load(f)

context2, run_log2, levels2 = shekina.run_process_table(table)
print("levels2:", levels2)
print("context2 keys:", list(context2.keys()))
import pandas as pd
print(pd.DataFrame(run_log2))
if "clean_df" in context2:
    display(context2["clean_df"].head())

levels2: [['s1'], ['s2'], ['s3']]
context2 keys: ['base_df', 'clean_df']
   id       step   status
0  s1  load_base       ok
1  s2    cleanup       ok
2  s3  skip_demo  skipped


,id,raw,clean
0,1,A,a
1,2,B,b


## Demo RIPS: pipeline de transformación
Este bloque muestra cómo, a partir de `base_df`, aplicamos transformaciones declarativas (Shekina → Alchemist) para aproximar un RIPS simplificado. Luego se puede escribir con Mirror.

In [ ]:
# Tabla canónica para un RIPS simplificado y ejecución con el mini-runner
rips_table = [
    {
        "id": "r1","step": "read_src","opcode": "mirror.select_df",
        "payload": {"data": [{"id": 1, "raw": "CITA-001"}, {"id": 2, "raw": "LAB-002"}]},
        "outputs": ["base_df"]
    },
    {
        "id": "r2","step": "normalize","depends_on": ["read_src"],
        "opcode": "alchemist.transmute",
        "payload": {
            "inputs": ["base_df"],
            "steps": [
                {"type": "CLEAN_IDENTIFIER", "src": "raw", "target": "clean", "pattern": "[^A-Za-z0-9]", "replacement": ""}
            ],
            "outputs": ["clean_df"]
        },
        "inputs": ["base_df"],
        "outputs": ["clean_df"]
    }
]

aleya2 = AleyaRunner()
shekina2 = ShekinaOrchestrator(registry=registry, aleya=aleya2)
context_r, log_r, levels_r = shekina2.run_process_table(rips_table)
print("levels_r:", levels_r)
print("context_r keys:", list(context_r.keys()))
import pandas as pd
print(pd.DataFrame(log_r))
if "clean_df" in context_r:
    display(context_r["clean_df"])